# **Agentic AI Tutor for French Civic Exam Preparation**

Name: Sabrina Palis  
Project Type: Applied AI / Educational Technology  
Format: Gradio-based Interactive System  



## 1. Problem Statement

## 1.1 Problem Statement

Preparing for the French civic integration exam is a high-stakes process, particularly for immigrants and non-native French speakers. The exam requires knowledge of abstract institutional, legal, and cultural concepts, which are often presented in dense or non-interactive formats.

Existing preparation methods rely heavily on memorization, static documents, or generic quizzes. These approaches fail to provide:

- Personalized feedback  
- Adaptive learning pathways  
- Clear explanations of complex civic concepts  
- Realistic simulation of exam conditions  

This creates a gap between **content exposure** and **actual understanding**, which can negatively impact exam performance and broader social integration.

This project addresses this gap by designing an **interactive AI tutor** that simulates exam conditions while providing targeted, bilingual feedback.


## 1.2. Target Users

The primary users of this system are:

- Immigrants preparing for the French civic integration exam  
- Non-native French speakers needing structured civic knowledge  
- Learners who benefit from interactive and guided learning  

The system is particularly designed for users who:

- Struggle with abstract institutional concepts  
- Need explanations in both French and English  
- Benefit from repeated testing and feedback loops  

The design prioritizes **clarity, accessibility, and progressive learning**.

## 2. Setup

In [ ]:
# Install dependencies
!pip install openai gradio

In [ ]:
# Imports
import random
import gradio as gr
from collections import Counter, defaultdict
from openai import OpenAI
# 🔑 Put your API key here
client = OpenAI(api_key="YOUR_API_KEY")

In [ ]:
# Global storage
FLAGGED_CASES = []
LAST_INTERACTION = {}

In [ ]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')

## 3. Dataset

## 3.1. Load Dataset

In [ ]:
#Add curriculum file path to python
import json
import random

with open('/content/drive/MyDrive/civic_tutor/french_civic_exam_curriculum_cleaned.json', 'r', encoding='utf-8') as f:
    CURRICULUM = json.load(f)

## 3.2. Dataset description

The system is based on a structured dataset of **360 civic questions and answers**, covering:

- Principles and values of the Republic  
- Institutional and political systems  
- Rights and duties  
- History, geography, and culture  
- Everyday life in French society  

Each entry includes:

- Question  
- Expected answer  
- Chapter  
- Optional tags (for thematic grouping)  

This dataset is stored locally as a JSON file, enabling:

- deterministic evaluation  
- controlled content scope  
- reproducibility of results  

## 3.3. Data preprocessing

In [ ]:
# ---------- Build flat dataset ----------

FLAT_CURRICULUM = []

for chapter, items in CURRICULUM.items():
    for item in items:
        FLAT_CURRICULUM.append({
            "chapter": chapter,
            "question": item["question"],
            "answer": item["answer"],
            "section": item.get("section", ""),
            "tags": item.get("tags", [])
        })

print(f"Dataset loaded: {len(FLAT_CURRICULUM)} questions")

## 4. Solution Overview

This project implements an AI-powered civic exam tutor using a hybrid architecture.

The system combines:

- a structured civic curriculum stored as JSON
- deterministic Python logic for question selection and scoring
- AI-generated bilingual feedback for explanation and tutoring
- a Gradio interface for interactive exam simulation

The main learning experience is a 40-question multiple-choice exam simulation. The user answers each question by selecting A, B, C, or D. The system evaluates the answer, gives feedback, tracks progress, and produces a final readiness report.

The design goal is to keep the system reliable, inexpensive, and usable while still offering an intelligent tutoring layer.

## 5. Architecture

The system is organized as a modular agentic workflow. Each component has a specific responsibility:

- `QuestionAgent`: selects questions from the curriculum
- `MCQAgent`: creates multiple-choice options
- `EvaluationAgent`: checks whether the selected answer is correct
- `FeedbackAgent`: generates bilingual tutoring feedback
- `AdaptationAgent`: identifies weak areas from mistakes
- `FinalReportAgent`: summarizes performance at the end of the exam

This modular design makes the system easier to maintain, test, and extend.

### 5.1 QuestionAgent

The `QuestionAgent` is responsible for selecting the exam questions.

In the current version, it randomly samples 40 questions from the full curriculum. This mirrors the real exam format more closely than chapter-by-chapter practice, because the learner must be ready to answer questions from any part of the civic curriculum.

In [ ]:
class QuestionAgent:
    def select_exam_questions(self, n=40):
        return random.sample(FLAT_CURRICULUM, n)

### 5.2 MCQAgent

The `MCQAgent` converts each curriculum question into a multiple-choice question.

The correct answer comes directly from the dataset. Distractors are selected from related answers, preferably from the same chapter and overlapping tags. This keeps the options more coherent and avoids mixing unrelated topics.

In [ ]:
class MCQAgent:
    def create_mcq(self, item):
        correct = item["answer"]
        item_tags = set(item.get("tags", []))

        # Best distractors: same chapter + overlapping tags
        tagged_pool = [
            x["answer"] for x in FLAT_CURRICULUM
            if x["chapter"] == item["chapter"]
            and x["answer"] != correct
            and item_tags.intersection(set(x.get("tags", [])))
        ]

        # Backup: same chapter
        chapter_pool = [
            x["answer"] for x in FLAT_CURRICULUM
            if x["chapter"] == item["chapter"]
            and x["answer"] != correct
        ]

        # Final backup: all curriculum
        global_pool = [
            x["answer"] for x in FLAT_CURRICULUM
            if x["answer"] != correct
        ]

        pool = tagged_pool if len(tagged_pool) >= 3 else chapter_pool
        pool = pool if len(pool) >= 3 else global_pool

        distractors = random.sample(pool, 3)

        options = distractors + [correct]
        random.shuffle(options)

        letters = ["A", "B", "C", "D"]
        option_map = dict(zip(letters, options))
        correct_letter = [k for k, v in option_map.items() if v == correct][0]

        return option_map, correct_letter

### 5.3 EvaluationAgent

The EvaluationAgent is responsible for determining whether the user's answer is correct.

Unlike the FeedbackAgent, this component is fully deterministic: it compares the selected letter (A, B, C, or D) with the correct option generated by the MCQAgent.

This design choice is important for two reasons:

- It guarantees consistent and reliable scoring  
- It avoids potential hallucinations from the language model  

By separating evaluation from explanation, the system ensures that correctness is always grounded in the dataset rather than inferred.

In [ ]:
class EvaluationAgent:
    def evaluate(self, user_letter, correct_letter):
        return user_letter == correct_letter

### 5.4 FeedbackAgent

The FeedbackAgent provides the pedagogical component of the system.

After each answer, it generates a structured explanation using a language model. This explanation includes:

- Whether the answer is correct or incorrect  
- The correct answer  
- A short explanation in French  
- A corresponding explanation in English  

Unlike the EvaluationAgent, this component leverages AI to produce contextual and adaptive feedback.

This hybrid approach balances:

- Deterministic correctness (for reliability)  
- AI-generated explanations (for learning effectiveness)  

As a result, the system moves beyond simple scoring and acts as an interactive tutor.

In [ ]:
class FeedbackAgent:
    def generate_feedback(self, item, user_letter, correct_letter, option_map, is_correct):
        user_answer = option_map.get(user_letter, "Aucune réponse")
        correct_answer = option_map[correct_letter]

        prompt = f"""
Tu es un tuteur bienveillant pour l'examen civique français.

Question :
{item["question"]}

Réponse correcte :
{correct_answer}

Réponse choisie par l'apprenant :
{user_letter}. {user_answer}

Résultat :
{"correct" if is_correct else "incorrect"}

Explique brièvement :
1. Si la réponse est correcte ou non.
2. Pourquoi la bonne réponse est correcte.
3. Pourquoi la réponse choisie est mauvaise si elle est incorrecte.
4. Donne une mini-explication en français.
5. Donne une mini-explication en anglais.

IMPORTANT :
- Reste factuel et précis
- N'invente pas d'information
- Si tu n'es pas sûr, dis "I may be mistaken"

Format :

{"🟢" if is_correct else "🔴"} **Résultat :**
...

🇫🇷 **Explication :**
...

🇬🇧 **Explanation:**
...
"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content

### 5.5 AdaptationAgent

The AdaptationAgent analyzes the user's mistakes during the exam session.

It uses the response history to identify weak areas, including:

- chapters with repeated errors  
- recurring thematic tags  
- concepts that may require additional review  

This component gives the system an adaptive dimension. Instead of treating all mistakes as isolated events, it looks for patterns across the session.

In the current MVP, adaptation is used mainly for diagnostic reporting at the end of the exam. In future versions, this agent could also influence question selection in real time by asking more questions from weak areas.

This makes the system more than a static quiz: it becomes a learning tool that can identify where the user needs targeted support.

In [ ]:
class AdaptationAgent:
    def weak_areas_report(self, history):
        wrong = [h for h in history if not h["is_correct"]]

        if not wrong:
            return "Aucune faiblesse détectée. Excellent résultat."

        chapter_counts = Counter(h["chapter"] for h in wrong)
        tag_counts = Counter(tag for h in wrong for tag in h.get("tags", []))

        report = "### 🎯 Points à revoir\n\n"

        report += "**Chapitres faibles :**\n"
        for chapter, count in chapter_counts.most_common():
            report += f"- {chapter}: {count} erreur(s)\n"

        if tag_counts:
            report += "\n**Thèmes faibles :**\n"
            for tag, count in tag_counts.most_common(8):
                report += f"- {tag}: {count} erreur(s)\n"

        return report

### 5.6 FinalReportAgent

The FinalReportAgent generates the final summary after the 40-question exam is completed.

It uses the session history to calculate:

- total number of questions answered  
- number of correct answers  
- final score percentage  
- overall readiness level  

It also summarizes weak areas by identifying chapters and tags where the user made mistakes.

This component is important because it transforms raw quiz results into actionable learning guidance. Instead of only telling the user their score, the system points them toward the specific areas they should review next.

The FinalReportAgent therefore supports the tutoring goal of the project: helping learners understand not only whether they passed, but also what they need to improve.

In [ ]:
class FinalReportAgent:
    def generate_report(self, history):
        total = len(history)
        correct = sum(1 for h in history if h["is_correct"])
        score = round((correct / total) * 100, 1) if total else 0

        if score >= 80:
            verdict = "✅ Très bon niveau — prêt pour l'examen."
        elif score >= 65:
            verdict = "🟡 Niveau correct — encore quelques points à revoir."
        else:
            verdict = "🔴 Révision nécessaire avant l'examen."

        weak_report = AdaptationAgent().weak_areas_report(history)

        return f"""
# 📊 Résultat final

**Score : {correct}/{total} — {score}%**

{verdict}

{weak_report}
"""

### 5.7 Exam Logic

The exam logic orchestrates the interaction between all agents and manages the user session.

Each session follows a structured sequence:

1. Initialize a set of 40 questions sampled from the full curriculum  
2. Present one question at a time  
3. Generate multiple-choice options  
4. Record the user's answer  
5. Evaluate correctness  
6. Generate feedback  
7. Update score and progress  
8. Move to the next question  

The system also maintains a history of responses, enabling:

- score computation  
- identification of weak areas  
- generation of a final performance report  

This coordination layer transforms independent components into a coherent learning system.

In [ ]:
# ---------- Exam state + logic ----------

question_agent = QuestionAgent()
mcq_agent = MCQAgent()
evaluation_agent = EvaluationAgent()
feedback_agent = FeedbackAgent()
final_report_agent = FinalReportAgent()


def start_exam():
    questions = question_agent.select_exam_questions(40)

    session = {
        "questions": questions,
        "index": 0,
        "history": [],
        "current_options": None,
        "current_correct_letter": None,
        "answered": False
    }

    question_text, session = show_current_question(session)

    return session, question_text, None, "", "Score : 0/0", current_progress(session), gr.update(interactive=False)


def show_current_question(session):
    index = session["index"]
    item = session["questions"][index]

    option_map, correct_letter = mcq_agent.create_mcq(item)

    session["current_options"] = option_map
    session["current_correct_letter"] = correct_letter
    session["answered"] = False

    question_text = f"""
# Question {index + 1}/40

**📚 {item["chapter"]}**

## {item["question"]}

---

**A.** {option_map["A"]}
**B.** {option_map["B"]}
**C.** {option_map["C"]}
**D.** {option_map["D"]}
"""

    return question_text, session


def submit_exam_answer(user_letter, session):
    if session is None:
        return (
            session,
            "Clique d'abord sur **Commencer l'examen**.",
            "",
            "Score : 0/0",
            "Progression : 0/40",
            user_letter,
            gr.update(interactive=False)
        )

    if user_letter is None:
        return (
            session,
            "Choisis A, B, C ou D.",
            "",
            current_score(session),
            current_progress(session),
            None,
            gr.update(interactive=False)
        )

    if session["answered"]:
        return (
            session,
            "Tu as déjà répondu à cette question. Clique sur **Question suivante**.",
            "",
            current_score(session),
            current_progress(session),
            None,
            gr.update(interactive=True)
        )

    index = session["index"]
    item = session["questions"][index]

    correct_letter = session["current_correct_letter"]
    option_map = session["current_options"]

    is_correct = evaluation_agent.evaluate(user_letter, correct_letter)

    session["history"].append({
        "question": item["question"],
        "answer": item["answer"],
        "chapter": item["chapter"],
        "tags": item.get("tags", []),
        "user_letter": user_letter,
        "correct_letter": correct_letter,
        "is_correct": is_correct
    })

    session["answered"] = True

    feedback = feedback_agent.generate_feedback(
        item=item,
        user_letter=user_letter,
        correct_letter=correct_letter,
        option_map=option_map,
        is_correct=is_correct
    )

    global LAST_INTERACTION

    LAST_INTERACTION = {
        "question": item["question"],
        "correct_answer": item["answer"],
        "user_letter": user_letter,
        "correct_letter": correct_letter,
        "feedback": feedback,
        "chapter": item["chapter"]
    }

    return (
        session,
        feedback,
        "",
        current_score(session),
        current_progress(session),
        None,
        gr.update(interactive=True)
    )


def next_question(session):
    if session is None:
        return (
            session,
            "Clique d'abord sur **Commencer l'examen**.",
            None,
            "",
            "Score : 0/0",
            "Progression : 0/40",
            gr.update(interactive=False)
        )

    if not session["answered"]:
        return (
            session,
            "Réponds d'abord à la question actuelle.",
            None,
            "",
            current_score(session),
            current_progress(session),
            gr.update(interactive=False)
        )

    session["index"] += 1

    if session["index"] >= 40:
        final_report = final_report_agent.generate_report(session["history"])
        return (
            session,
            final_report,
            None,
            "",
            current_score(session),
            "Progression : 40/40",
            gr.update(interactive=False)
        )

    question_text, session = show_current_question(session)

    return (
        session,
        question_text,
        None,
        "",
        current_score(session),
        current_progress(session),
        gr.update(interactive=False)
    )

def flag_response():
    if not LAST_INTERACTION:
        return "Aucune réponse à signaler."

    FLAGGED_CASES.append(LAST_INTERACTION.copy())

    return "⚠️ Réponse signalée pour révision."

def explain_again():
    if not LAST_INTERACTION:
        return "Aucune réponse à réexpliquer."

    item_question = LAST_INTERACTION["question"]
    correct_answer = LAST_INTERACTION["correct_answer"]

    prompt = f"""
Explique simplement cette question du test civique français.

Question:
{item_question}

Réponse:
{correct_answer}

Donne une explication courte en français puis en anglais.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

def current_score(session):
    if session is None or not session["history"]:
        return "Score : 0/0"

    total = len(session["history"])
    correct = sum(1 for h in session["history"] if h["is_correct"])

    return f"Score : {correct}/{total}"

def current_progress(session):
    if session is None:
        return "Progression : 0/40"
    return f"Progression : {session['index'] + 1}/40"

### 5.8 Governance Layer

The system includes a lightweight governance layer designed to improve reliability, transparency, and user control.

This layer does not generate questions or evaluate answers. Instead, it supports the learning process by allowing users to:

- flag responses they consider unclear or incorrect  
- request additional explanations  
- interact with the system beyond passive evaluation  

These mechanisms are particularly important in a high-stakes context such as civic exam preparation, where incorrect or unclear explanations may affect understanding.

### 5.9 Human-in-the-Loop Review

The system integrates a basic Human-in-the-Loop mechanism.

Users can flag a response when they disagree with the feedback or find it unclear. Flagged interactions are stored locally during the session and include:

- the question  
- the expected answer  
- the selected answer  
- the generated feedback  
- the associated chapter  

This creates a simple audit trail that can be used to:

- identify weaknesses in the system  
- improve prompts or explanations  
- support future expert review  

Although minimal, this mechanism introduces user agency and accountability into the system.

### 5.10 Responsible Feedback Design

The FeedbackAgent includes prompt-level constraints to reduce the risk of misleading explanations.

It is instructed to:

- remain factual and precise  
- avoid inventing information  
- provide neutral and respectful explanations  
- acknowledge uncertainty when necessary  

This is particularly important because explanations are generated by a language model, while correctness is determined deterministically.

By separating evaluation from explanation and constraining the latter, the system improves reliability while preserving pedagogical value.

## 6. User Interface

The user interface is implemented using Gradio and designed as a focused exam simulation environment.

The interface includes:

- a start button to initialize the exam  
- a question display area  
- multiple-choice selection (A, B, C, D)  
- a validation button  
- a next-question button  
- score and progress indicators  
- AI-generated feedback  
- a flagging button for Human-in-the-Loop review  
- a re-explanation button for additional support  

Several design choices improve usability:

- larger font size and high contrast for readability  
- controlled interaction flow (users must answer before proceeding)  
- clear separation between question, answer, and feedback  
- minimal interface complexity  

The interface is designed to simulate an exam environment while preserving interactive learning.

In [ ]:
with gr.Blocks(css="""
.gradio-container {
    font-size: 20px !important;
    color: black !important;
}

textarea, input {
    font-size: 20px !important;
    color: black !important;
}

.gr-markdown {
    font-size: 20px !important;
    color: black !important;
    line-height: 1.6 !important;
}

label, .wrap {
    font-size: 20px !important;
    color: black !important;
}
""") as app:

    gr.Markdown("# 🇫🇷 AI Civic Tutor — Simulation d'examen")
    progress_box = gr.Markdown("Progression : 0/40")

    session_state = gr.State()

    start_btn = gr.Button("▶️ Commencer l'examen de 40 questions")

    question_box = gr.Markdown()

    answer_choice = gr.Radio(
        choices=["A", "B", "C", "D"],
        label="Choisis ta réponse",
        interactive=True
    )

    with gr.Row():
        submit_btn = gr.Button("✅ Valider la réponse", scale=2)
        next_btn = gr.Button("➡️ Question suivante", scale=1, interactive=False)

    with gr.Row():
        flag_btn = gr.Button("⚠️ Signaler une réponse")
        explain_btn = gr.Button("🔁 Réexpliquer")

    feedback_box = gr.Markdown()
    score_box = gr.Textbox(label="Score", value="Score : 0/0", lines=1)

    start_btn.click(
        fn=start_exam,
        inputs=[],
        outputs=[
            session_state,
            question_box,
            answer_choice,
            feedback_box,
            score_box,
            progress_box,
            next_btn
        ]
    )

    submit_btn.click(
        fn=submit_exam_answer,
        inputs=[answer_choice, session_state],
        outputs=[
            session_state,
            feedback_box,
            question_box,
            score_box,
            progress_box,
            answer_choice,
            next_btn
        ]
    )

    next_btn.click(
        fn=next_question,
        inputs=[session_state],
        outputs=[
            session_state,
            question_box,
            answer_choice,
            feedback_box,
            score_box,
            progress_box,
            next_btn
        ]
    )

    flag_btn.click(
        fn=flag_response,
        inputs=[],
        outputs=[feedback_box]
    )

    explain_btn.click(
        fn=explain_again,
        inputs=[],
        outputs=[feedback_box]
    )

app.launch(debug=True)

### 6.1 User Interaction Flow

The interaction flow is structured as follows:

1. Start the exam session  
2. Read the question and answer options  
3. Select an answer (A, B, C, or D)  
4. Validate the answer  
5. Read the bilingual feedback  
6. Optionally flag the response or request a new explanation  
7. Proceed to the next question  

This structure combines assessment and learning, allowing users to both test and improve their understanding.

## 7. Evaluation

The system evaluates user performance through a combination of quantitative and qualitative measures.

Quantitative metrics include:

- Total number of correct answers  
- Score percentage over 40 questions  
- Progress tracking  

Qualitative evaluation includes:

- Identification of weak chapters  
- Analysis of recurring errors  
- Bilingual feedback for conceptual understanding  

At the end of the exam, the system generates a summary report that provides:

- A readiness assessment  
- Key areas for improvement  

This dual evaluation approach ensures that the system supports both performance measurement and learning.

## 8. Limitations

While the system provides a functional and effective learning experience, several limitations remain:

- The adaptation is limited to a single session (no persistent user memory)  
- Difficulty does not dynamically adjust based on performance  
- Distractor generation is heuristic-based and may not always be optimal  
- Feedback quality depends on the language model  

Additionally, the system does not yet implement:

- spaced repetition  
- long-term progress tracking  
- personalized learning paths  

These limitations highlight areas for future improvement.

## 9. Future Work

Several improvements can extend the system into a more advanced agentic learning platform:

- Persistent user profiles and progress tracking  
- Adaptive difficulty based on performance  
- Spaced repetition for long-term retention  
- Improved distractor generation using semantic similarity  
- Deployment as a public web application (e.g., Hugging Face Spaces)  
- Integration of speech for oral exam simulation  

In the longer term, this project can evolve into a fully governed agentic system capable of:

- modeling learner behavior  
- optimizing learning strategies  
- supporting high-stakes educational contexts  

These extensions align with broader research directions in AI-assisted education and decision systems.

## 10. Conclusion

This project demonstrates how an AI system can be applied to a real-world, high-stakes educational context: preparation for the French civic integration exam.

By combining:

- a structured and domain-specific dataset  
- deterministic evaluation logic  
- AI-generated bilingual feedback  
- weak-area analysis  
- a Human-in-the-Loop mechanism  
- a lightweight governance layer  

the system moves beyond a traditional quiz format and becomes an **interactive learning environment**.

A key design principle of the project is the separation between:

- **evaluation**, which is deterministic and grounded in the dataset  
- **explanation**, which is generated by a language model under controlled constraints  

This separation improves reliability while preserving the flexibility and pedagogical value of AI-generated feedback.

The introduction of a governance layer, even in a minimal form, highlights important considerations in AI system design:

- transparency through logging and traceability  
- user agency through feedback and flagging  
- responsibility through prompt constraints and uncertainty awareness  

Although the system remains an MVP, it already reflects a broader direction:

> designing AI systems that operate in uncertain environments while remaining interpretable, controllable, and aligned with user needs.

In this sense, the project is not only an educational tool, but also a step toward **governed, modular AI systems** that support decision-making and learning in real-world contexts.